In [5]:
import re
import pandas as pd
import os

# Function to extract Title, ATOM PARAMETERS, and Cell parameters sections from the content
def extract_sections(content):
    data = {}
    title_match = re.search(r"Title:\s*(.+)", content)
    data['Title'] = title_match.group(1).strip() if title_match else None

    atom_params_match = re.search(r"==> ATOM PARAMETERS:\s*\n((?:.*\n)+?)(?=\n\s*==>)", content)
    data['ATOM PARAMETERS'] = atom_params_match.group(1).strip() if atom_params_match else None

    cell_params_match = re.search(r"Cell parameters\s*:\s*\n((?:\s*\d+\.\d+\s*\d+\.\d+\s*\n)+)", content)
    data['Cell parameters'] = cell_params_match.group(1).strip() if cell_params_match else None
    
    overall_tem_factor_match = re.search(r"=> Overall tem\. factor\s*:\s*(\d+\.\d+)\s+(\d+\.\d+)", content)
    data['Overall tem. factor'] = overall_tem_factor_match.group(0).strip() if overall_tem_factor_match else None

    r_factors_match = re.search(r"=> Rp:\s*(\d+\.\d+)\s+Rwp:\s*(\d+\.\d+)\s+Rexp:\s*(\d+\.\d+)\s+Chi2:\s*(\d+\.\d+)", content)
    data['R factors'] = r_factors_match.group(0).strip() if r_factors_match else None

    bragg_r_factor_match = re.search(r"=> Bragg R-factor:\s*(\d+\.\d+)\s+Vol:\s*(\d+\.\d+)\(\s*\d+\.\d+\)\s+Fract\(%\):\s*(\d+\.\d+)\(\s*\d+\.\d+\)", content)
    data['Bragg R-factor'] = bragg_r_factor_match.group(0).strip() if bragg_r_factor_match else None
    
    return data

# Function to organize the title into separate columns
def organize_title(title):
    if title:
        parts = title.split('-')
        if len(parts) == 2:
            running_number = parts[0].strip()
            temperature = parts[1].strip().replace('K', '').replace(' ', '-')
            if temperature.endswith('-'):
                temperature = temperature[:-1]
            return running_number, temperature
    return None, None

# Function to parse ATOM PARAMETERS and create a DataFrame
def parse_atom_parameters(atom_params):
    lines = atom_params.split('\n')
    columns = ['Name', 'x', 'sx', 'y', 'sy', 'z', 'sz', 'B', 'sB', 'occ', 'socc', 'Mult']
    data = []
    for line in lines[1:]:  # Skip the header line
        parts = re.split(r'\s+', line.strip())
        if len(parts) == 12:
            data.append(parts)
    
    atom_df = pd.DataFrame(data, columns=columns)
    
    organized_data = {}
    
    for _, row in atom_df.iterrows():
        name = row['Name']
        organized_data[f'x({name})'] = row['x']
        organized_data[f'ESD(x({name}))'] = row['sx']
        organized_data[f'y({name})'] = row['y']
        organized_data[f'ESD(y({name}))'] = row['sy']
        organized_data[f'z({name})'] = row['z']
        organized_data[f'ESD(z({name}))'] = row['sz']
        organized_data[f'B({name})'] = row['B']
        organized_data[f'ESD(B({name}))'] = row['sB']
        organized_data[f'Occupation factor({name})'] = row['occ']
        organized_data[f'ESD(occupation factor({name}))'] = row['socc']
    
    return pd.DataFrame([organized_data])

# Function to parse and organize Cell parameters
def organize_cell_parameters(cell_params):
    lines = cell_params.strip().split('\n')
    organized_data = {}
    
    if len(lines) == 6:
        organized_data['Unit cell parameter a'] = lines[0].strip().split()[0]
        organized_data['Unit cell parameter a ESD'] = lines[0].strip().split()[1]
        organized_data['Unit cell parameter b'] = lines[1].strip().split()[0]
        organized_data['Unit cell parameter b ESD'] = lines[1].strip().split()[1]
        organized_data['Unit cell parameter c'] = lines[2].strip().split()[0]
        organized_data['Unit cell parameter c ESD'] = lines[2].strip().split()[1]
        organized_data['Unit cell parameter alpha'] = lines[3].strip().split()[0]
        organized_data['Unit cell parameter alpha ESD'] = lines[3].strip().split()[1]
        organized_data['Unit cell parameter beta'] = lines[4].strip().split()[0]
        organized_data['Unit cell parameter beta ESD'] = lines[4].strip().split()[1]
        organized_data['Unit cell parameter gamma'] = lines[5].strip().split()[0]
        organized_data['Unit cell parameter gamma ESD'] = lines[5].strip().split()[1]
    
    return pd.DataFrame([organized_data])

# Function to parse and organize Overall tem. factor
def organize_overall_tem_factor(overall_tem_factor):
    parts = overall_tem_factor.replace('=> Overall tem. factor  :', '').strip().split()
    if len(parts) == 2:
        return pd.Series({'Bover': parts[0], 'ESD(Bover)': parts[1]})
    return pd.Series({'Bover': None, 'ESD(Bover)': None})

# Function to parse and organize R factors
def organize_r_factors(r_factors):
    parts = r_factors.replace('=> Rp:', '').replace('Rwp:', '').replace('Rexp:', '').replace('Chi2:', '').split()
    if len(parts) == 4:
        return pd.Series({'Rp': parts[0], 'Rwp': parts[1], 'Chi^2': parts[3]})
    return pd.Series({'Rp': None, 'Rwp': None, 'Chi^2': None})

# Function to parse and organize Bragg R-factor
def organize_bragg_r_factor(bragg_r_factor):
    match = re.search(
        r'=> Bragg R-factor:\s+([\d.]+)\s+Vol:\s+([\d.]+)\s*\(\s*([\d.]+)\s*\)\s+Fract\(%\):\s+([\d.]+)\s*\(\s*([\d.]+)\s*\)',
        bragg_r_factor
    )
    if match:
        RBragg, V, ESD_V, w, ESD_w = match.groups()
        return pd.Series({
            'RBragg': RBragg,
            'Unit cell volume V': V,
            'ESD(V)': ESD_V,
            'Phase fraction w': w,
            'ESD(w)': ESD_w
        })
    return pd.Series({
        'RBragg': None,
        'Unit cell volume V': None,
        'ESD(V)': None,
        'Phase fraction w': None,
        'ESD(w)': None
    })

# Function to clean parentheses from the DataFrame
def remove_parentheses(df):
    df_cleaned = df.applymap(lambda x: re.sub(r'[()]', '', str(x)))
    return df_cleaned

# Directory containing .sum files
sum_files_directory = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\Large Sum File\Sum\Sum'

# Create a new directory to save CSV files
output_directory = os.path.join(sum_files_directory, 'CSV_Output_Step_1')
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Get list of all .sum files in the directory
sum_files = [f for f in os.listdir(sum_files_directory) if f.endswith('.sum')]

# Process each .sum file and save to CSV
for sum_file in sum_files:
    file_path = os.path.join(sum_files_directory, sum_file)
    
    # Read the content of the .sum file
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        content = file.read()
    
    # Extract data sections
    extracted_data = extract_sections(content)

    # Create DataFrame
    df = pd.DataFrame([extracted_data])

    # Organize the 'Title' column
    df[['Running Number of Data File', 'Temperature of measurement']] = df['Title'].apply(
        lambda x: pd.Series(organize_title(x))
    )
    df.drop(columns=['Title'], inplace=True)

    # Parse and merge ATOM PARAMETERS
    atom_parameters_df = df['ATOM PARAMETERS'].apply(parse_atom_parameters)
    atom_parameters_df = pd.concat(atom_parameters_df.values).reset_index(drop=True)
    df.drop(columns=['ATOM PARAMETERS'], inplace=True)
    df = pd.concat([df, atom_parameters_df], axis=1)

    # Parse and merge Cell parameters
    cell_parameters_df = df['Cell parameters'].apply(organize_cell_parameters)
    cell_parameters_df = pd.concat(cell_parameters_df.values).reset_index(drop=True)
    df.drop(columns=['Cell parameters'], inplace=True)
    df = pd.concat([df, cell_parameters_df], axis=1)

    # Parse and merge Overall tem. factor
    df[['Bover', 'ESD(Bover)']] = df['Overall tem. factor'].apply(organize_overall_tem_factor)
    df.drop(columns=['Overall tem. factor'], inplace=True)

    # Parse and merge R factors
    df[['Rp', 'Rwp', 'Chi^2']] = df['R factors'].apply(organize_r_factors)
    df.drop(columns=['R factors'], inplace=True)

    # Parse and merge Bragg R-factor
    df[['RBragg', 'Unit cell volume V', 'ESD(V)', 'Phase fraction w', 'ESD(w)']] = df['Bragg R-factor'].apply(
        organize_bragg_r_factor
    )
    df.drop(columns=['Bragg R-factor'], inplace=True)

    # Clean the DataFrame
    df_cleaned = remove_parentheses(df)

    # Save DataFrame to CSV in the new directory
    output_csv_path = os.path.join(output_directory, sum_file.replace('.sum', '.csv'))
    df_cleaned.to_csv(output_csv_path, index=False)

print(f"CSV files have been saved to {output_directory}")


CSV files have been saved to C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\Large Sum File\Sum\Sum\CSV_Output_Step_1
